In [ ]:
!pip install opencv-python-headless numpy pillow mediapipe

In [ ]:
import cv2  # Библиотека для работы с изображениями и видео
import numpy as np  # Библиотека для работы с массивами данных
import mediapipe as mp  # Библиотека для компьютерного зрения и распознавания лиц
from PIL import Image  # Библиотека для работы с изображениями
from google.colab import files  # Библиотека для загрузки файлов в Colab
import matplotlib.pyplot as plt  # Библиотека для отображения изображений

# Инициализация MediaPipe для обнаружения лиц
mp_face_detection = mp.solutions.face_detection  # Модуль MediaPipe для обнаружения лиц
mp_drawing = mp.solutions.drawing_utils  # Модуль для рисования на изображениях

# Функция для обнаружения лиц на изображении
def detect_faces(image):
    img_array = np.array(image)  # Преобразование изображения из PIL в массив NumPy
    img_rgb = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)  # Преобразование цвета из BGR в RGB

    # Инициализация модели для обнаружения лиц
    with mp_face_detection.FaceDetection(min_detection_confidence=0.2) as face_detection:

#Используется контекстный менеджер (with ... as ...). Это означает, что после выполнения блока кода объект face_detection будет автоматически закрыт (удалён), освобождая ресурсы.

#mp_face_detection.FaceDetection(...) создаёт объект детектора лиц из библиотеки MediaPipe.

#Аргумент min_detection_confidence=0.2 указывает минимальный порог уверенности модели, с которым она считает, что нашла лицо.

#0.2 (или 20%) – означает, что модель будет считать найденные лица корректными, если она уверена в этом минимум на 20%.

#Чем выше значение (например, 0.8 или 80%), тем более точным будет обнаружение, но при этом модель может пропускать некоторые лица.

#Переменная face_detection теперь содержит объект, с помощью которого можно обрабатывать изображения и находить на них лица.




        results = face_detection.process(img_rgb)  # Обработка изображения для обнаружения лиц

#Вызывается метод .process(img_rgb), который обрабатывает переданное изображение (img_rgb) и ищет на нём лица.

#img_rgb – это исходное изображение, которое мы предварительно конвертировали из BGR в RGB (так требует MediaPipe).

#Результаты обнаружения лиц сохраняются в переменную results.

#results – это специальный объект MediaPipe, который содержит координаты найденных лиц и информацию о них.


#Что будет храниться в results?
#Если на изображении нашлись лица, то results будет содержать:

#results.detections – список объектов, каждый из которых описывает одно лицо.

#В каждом объекте хранятся данные о лице:

#Координаты местоположения (центр, границы лица).

#Оценка уверенности (насколько алгоритм уверен, что это лицо).

#Дополнительные параметры (например, наличие солнцезащитных очков).

#Если лиц не найдено, то results.detections будет равно None.


    if results.detections:  # Если лица обнаружены
        for detection in results.detections:  # Проход по всем обнаруженным лицам
            bboxC = detection.location_data.relative_bounding_box  # Получение координат прямоугольника
            ih, iw, _ = img_array.shape  # Получение размеров изображения
#img_array.shape — возвращает размеры изображения в виде кортежа:
#ih, iw, _ — это распаковка кортежа:
#ih (image height) — высота изображения (в пикселях).
#iw (image width) — ширина изображения (в пикселях).
#_ — третье значение (количество цветовых каналов, например, 3 для RGB) нам не нужно, поэтому его заменяют _ (переменная-заглушка).


            # Преобразование относительных координат в абсолютные
            bbox = int(bboxC.xmin * iw), int(bboxC.ymin * ih), \
                   int(bboxC.width * iw), int(bboxC.height * ih)
           # bboxC = detection.location_data.relative_bounding_box
#detection.location_data.relative_bounding_box — содержит координаты прямоугольника вокруг лица в относительных значениях (от 0 до 1).

#Эти координаты представляют долю от ширины и высоты изображения.

#Здесь каждая координата умножается на размеры изображения (iw, ih), чтобы получить абсолютные пиксельные значения(то есть уже в реальных пикселях, а не от 0 до 1).




            # Рисование прямоугольника вокруг лица
            cv2.rectangle(img_array, bbox, (255, 0, 0), 2)
#img_array — изображение, на котором рисуем.
#bbox — координаты прямоугольника в формате (x, y, width, height), но cv2.rectangle ожидает (x1, y1, x2, y2), поэтому используется:
#cv2.rectangle(img_array, (bbox[0], bbox[1]), (bbox[0] + bbox[2], bbox[1] + bbox[3]), (255, 0, 0), 2)
#(bbox[0], bbox[1]) — верхний левый угол (xmin, ymin).
#(bbox[0] + bbox[2], bbox[1] + bbox[3]) — нижний правый угол (xmax, ymax).
#(255, 0, 0) — синий цвет (OpenCV использует BGR).
#2 — толщина линии.
#Пример
#Если bbox = (256, 72, 384, 288), то рисуется прямоугольник от (256, 72) до (256+384, 72+288), т.е. от (256, 72) до (640, 360).


            # Добавление текста "Face" над прямоугольником
            cv2.putText(img_array, 'Face', (bbox[0], bbox[1] - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
#img_array — изображение, на котором пишем текст.
#'Face' — сам текст.

#(bbox[0], bbox[1] - 10) — координаты начала текста:
#bbox[0] — X-координата (начинается от левого верхнего угла лица).
#bbox[1] - 10 — Y-координата (чтобы текст был чуть выше прямоугольника).

#В OpenCV (и вообще в компьютерном зрении) начало координат (0,0) находится в левом верхнем углу изображения:

#Ось X (ширина) → растёт вправо.
#Ось Y (высота) ↓ растёт вниз.


#cv2.FONT_HERSHEY_SIMPLEX — шрифт.
#0.9 — размер шрифта.
#(255, 0, 0) — цвет (синий).
#2 — толщина букв.

    return img_array  # Возвращение обработанного изображения

# Функция для загрузки изображения
def upload_image(title): #title нигде не используется в коде, поэтому он не играет роли в выполнении функции (возможно, это задел на будущее, например, для вывода заголовка перед загрузкой).
    uploaded = files.upload()  # Загрузка файла
   # files.upload() – это метод из библиотеки Google Colab, который открывает окно загрузки файла.

#Он позволяет пользователю выбрать локальный файл и загрузить его в Colab.

#files.upload() возвращает словарь, где:

#Ключи – имена загруженных файлов (строки).

#Значения – бинарные данные (файловое содержимое).
    for filename in uploaded.keys():  # Проход по загруженным файлам
        return Image.open(filename)  # Открытие изображения и возврат его в формате PIL. Image.open(filename) – это метод из библиотеки PIL (Pillow), который открывает изображение.

In [ ]:
# Загрузка изображения для обработки
print("Загрузите изображение для обработки")  # Подсказка пользователю
uploaded_image = upload_image("Изображение для обработки")  # Вызов функции для загрузки изображения
#Что делает upload_image()?

#Открывает окно загрузки файлов (в Google Colab).

#Пользователь выбирает файл (например, face.jpg).

#Файл загружается и открывается в формате PIL.Image.

#uploaded_image теперь содержит изображение.



# Обработка изображения
result_img = detect_faces(uploaded_image)  # Обработка изображения для обнаружения лиц


# Вывод обработанного изображения с лицами и подписями
plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))  # Преобразование изображения из BGR в RGB и отображение
plt.axis('off')  # Отключение отображения осей, чтобы картинка выглядела чище.
plt.show()  # Отображение изображения c прямоугольниками вокруг лиц



#cv2.cvtColor меняет порядок цветов, чтобы изображение корректно отобразилось.

#plt.imshow(...)

#Показывает обработанное изображение с прямоугольниками вокруг лиц.



In [ ]:
pip install gtts

In [ ]:
from gtts import gTTS  # Импорт библиотеки gTTS для преобразования текста в речь
import IPython.display as ipd  # Импорт модуля IPython для воспроизведения аудио

# Функция для чтения текста из файла
def read_text_from_file(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as file:  # Открытие файла с заданным путем в режиме чтения с кодировкой UTF-8
            return file.read()  # Возвращение содержимого файла как строки
    except FileNotFoundError:
        return "Файл не найден."  # Возвращение сообщения об ошибке, если файл не найден
    except IOError:
        return "Ошибка чтения файла."  # Возвращение сообщения об ошибке в случае других проблем с чтением файла

# Функция для преобразования текста в аудио
def text_to_speech(text, lang='ru'):
    tts = gTTS(text=text, lang=lang, slow=False)  # Создание объекта gTTS для преобразования текста в речь с указанным языком и скоростью
    audio_file = 'output.mp3'  # Имя файла для сохранения сгенерированного аудио

#Сохраняет файл output.mp3 в текущей рабочей директории.

#Как узнать, где именно?
#В Google Colab(а код был разработан именно для Google Colab) текущая директория по умолчанию — /content/.

#Значит, файл output.mp3 сохранится в /content/output.mp3.


    tts.save(audio_file)  # Сохранение аудио в файл с указанным именем
    return audio_file  # Возвращение имени сохраненного аудиофайла

# Путь к текстовому файлу
file_path = '/content/textfile.txt'  # Определение пути к текстовому файлу


# Чтение текста из файла
text = read_text_from_file(file_path)  # Вызов функции для чтения текста из файла

# Преобразование текста в аудио
audio_file = text_to_speech(text)  # Вызов функции для преобразования текста в аудио и получение пути к аудиофайлу

# Воспроизведение аудиофайла в Jupyter Notebook или Google Colab
ipd.display(ipd.Audio(audio_file))  # Использование IPython для воспроизведения аудиофайла


In [ ]:
from google.colab import files
#Что делает эта строка?

#Импортирует модуль files, который предоставляет методы для работы с файлами в Google Colab.

#Этот модуль позволяет загружать файлы на сервер Colab и скачивать их обратно на твой локальный компьютер.



# Укажите путь к вашему аудиофайлу
audio_file_path = 'output.mp3'

# Загрузка файла
files.download(audio_file_path)
#Что делает эта строка?

#Запускает скачивание файла output.mp3 на локальный компьютер.

#Открывается окно браузера, и файл начинает загружаться как обычный скачиваемый файл.


In [ ]:
# Установка необходимых библиотек
!pip install git+https://github.com/openai/whisper.git
#! в начале строки означает, что эта команда выполняется в командной строке (терминале), а не в Python.

#pip install — команда для установки Python-библиотек.

#git+https://github.com/openai/whisper.git — устанавливает пакет whisper прямо из репозитория GitHub(репозиторий называется whisper.git, программа извлекает его, ищет в самом репозитории установочный файл и уже с помощью него скачивает сам whisper).

# Что такое Whisper?

#Это модель OpenAI для автоматического распознавания речи.

#Она умеет транскрибировать аудио и даже переводить речь с одного языка на другой.



!sudo apt update && sudo apt install ffmpeg


#Что здесь происходит?

#!sudo apt update

#sudo — выполняет команду от имени администратора (root).

#apt update — обновляет список пакетов в системе Ubuntu (Google Colab работает на базе Ubuntu Linux).

#Это нужно, чтобы перед установкой пакет ffmpeg был свежей версии.

#&& Означает "выполнить следующую команду только если предыдущая прошла успешно".

#sudo apt install ffmpeg Устанавливает ffmpeg, инструмент для работы с аудио и видео.

#ffmpeg нужен Whisper, потому что он умеет конвертировать аудиофайлы в нужный формат для обработки.

#Что такое ffmpeg?

#Это мощный инструмент для обработки аудио и видео.

#Whisper использует ffmpeg, чтобы загружать, декодировать и конвертировать(преобразовывать в другой формат) аудиофайлы перед их распознаванием.

In [ ]:
!whisper "/content/output.mp3" —model medium
# Когда в Colab или Jupyter пишем "!" перед командой, Python передаёт её в терминал (Shell).


#Эта строка выполняется в Google Colab или в терминале Jupyter Notebook (из-за ! в начале). Она запускает OpenAI Whisper для автоматического распознавания речи из аудиофайла.

#--model указывает, какую модель Whisper использовать.

#medium — это средняя по размеру модель. Whisper предлагает 5 моделей:

#tiny (самая лёгкая, но менее точная)

#base

#small

#medium (оптимальный баланс точности и скорости)

#large (самая мощная, но медленная)